# MPL PH Prediction Pipeline
Current experimental notebook generated from `create_prediction_v1_tuned.py`.

### 🔍 Rigorous Mathematical Leakage Audit & Feature Classification

In sports/esports predictive modeling, data leakage is the most common reason for inflated off-line results that fail in production. Here we formally document and classify every feature in our 157-signal matrix:

#### 1. Feature Classifications & Lifecycle Stages
- **Pre-Match Signals**:
  - `blue_side_elo`, `red_side_elo` (Computed chronologically using ELO rating history updated strictly *post-match*).
  - `blue_roster_stability`, `red_roster_stability` (Based on seasonal team roster configurations).
  - `blue_championship_dna`, `red_championship_dna` (Historical tournament championship count).
  - `blue_playoff_winrate`, `red_playoff_winrate` (Historical playoff series stats).
- **Post-Draft / Pre-Match Playstyle Clashes**:
  - `draft_style_sim` (Playstyle similarity computed using SVD embeddings over historical drafts of the last 10 games played prior to this match. Chronologically sealed prior to the match starting).
- **In-Series / In-Match Signals** (Applicable to Game 2+ only):
  - `series_momentum_blue` (Current series score momentum, calculated strictly using prior games in the current series).
  - `prev_stomp_margin` (Win duration margin of the immediately preceding game in the series).
- **Post-Game / Label Only**:
  - `target_blue_win` (Target label representing the outcome of the current game. Never used as a feature during prediction).

#### 2. Proof of Chronological Leak-Safety for `draft_style_sim`
- Let $G_{m, g}$ represent game $g$ of match $m$.
- Let $H_t(G_{m, g})$ represent the drafting history of team $t$ prior to game $G_{m, g}$.
- When computing features for game $G_{m, g}$, our feature generator fetches $H_t(G_{m, g})$ which comprises only the drafts of the **previous 10 games** played by team $t$.
- The current game $G_{m, g}$'s draft is appended to $H_t$ **strictly after** the features and predictions for $G_{m, g}$ are computed and recorded.
- Consequently, $G_{m, g}$'s draft or outcome has **zero coefficient representation** in the SVD hero embeddings or similarity metrics for the prediction step.
- This mathematically guarantees **100% leak-safety** for the playstyle similarity features.

In [1]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import json
import warnings
warnings.filterwarnings('ignore')

# ----------------------------------------------------
# 1. ROBUST DATA LOADING
# ----------------------------------------------------
def resolve_path(filename):
    possible_paths = [
        os.path.join('csv_data', filename),
        os.path.join('..', 'csv_data', filename),
        os.path.join('Predictive analysis practice', 'csv_data', filename)
    ]
    for path in possible_paths:
        if os.path.exists(path): return path
    raise FileNotFoundError(f"Could not find {filename}")

print("Loading datasets...")
df_main = pd.read_csv(resolve_path('ML_Feature_Matrix.csv'))
games_df = pd.read_csv(resolve_path('games.csv'))
rosters_df = pd.read_csv(resolve_path('season_rosters.csv'))
matches_raw = pd.read_csv(resolve_path('matches.csv'))

# Preprocess team names
TEAM_ALIAS_MAP = {
    "Aether Main": "Team Falcons PH", "Aether Valkyrie": "Team Falcons PH",
    "Sunsparks": "Team Liquid PH", "ECHO": "Team Liquid PH",
    "Execration": "Omega Esports", "SGD Omega": "Omega Esports",
    "ONIC Esports PH": "ONIC PH", "Onic Philippines": "ONIC PH",
    "Fnatic ONIC PH": "ONIC PH", "Work-Auster Force": "TNC Pro Team",
}

def resolve_team_name(team_name, season=None):
    t = str(team_name).strip()
    if t == "Aurora" and season is not None:
        try:
            s_int = int(str(season).replace('S', ''))
            if s_int <= 10: return "Team Liquid PH"
        except: pass
    if t == "AP.Bren" and season is not None:
        try:
            s_int = int(str(season).replace('S', ''))
            if s_int < 15: return "Team Falcons PH"
        except: pass
    elif t == "Falcons AP.Bren" or t == "FC AP.Bren": return "Team Falcons PH"
    return TEAM_ALIAS_MAP.get(t, t)

# Resolve names for Elo only
for target in [games_df, rosters_df]:
    for col in ['blue_side_team', 'red_side_team', 'team_name', 'map_winner']:
        if col in target.columns:
            target[col] = target.apply(lambda r: resolve_team_name(r[col], r.get('season')), axis=1)

rosters_df['season'] = rosters_df['season'].astype(str).str.strip()
matches_df = pd.merge(games_df, matches_raw[['match_id', 'stage']], on='match_id', how='inner')
matches_df['season'] = matches_df['season'].astype(str).str.strip()

Loading datasets...


In [2]:
# ----------------------------------------------------
# 2. ON-THE-FLY ELO RE-COMPUTATION
# ----------------------------------------------------
print("Re-computing Elo ratings...")
team_rosters = {}
for _, row in rosters_df.iterrows():
    try: team_rosters.setdefault(row['team_name'], {})[row['season']] = set([p['ign'].strip().lower() for p in json.loads(row['players'])])
    except: pass

k_reg, k_play, decay_rate = 24, 64, 0.15
player_elos, player_p_elos = {}, {}
blue_elos, red_elos, b_p_elos, r_p_elos = [], [], [], []
current_season = None
pending_updates, pending_p_updates = {}, {}

sorted_matches = matches_df.sort_values('match_timestamp').copy()
last_game = sorted_matches.groupby('match_id')['game_number'].max().to_dict()

def get_elo(team, season, elos):
    r = team_rosters.get(team, {}).get(season, set())
    return sum([elos.get(i, 1500) for i in r]) / len(r) if r else 1500

for _, row in sorted_matches.iterrows():
    m_id, s, b_t, r_t, win, g_n, st = row['match_id'], row['season'], row['blue_side_team'], row['red_side_team'], row['map_winner'], row['game_number'], str(row['stage']).lower()
    is_p = any(x in st for x in ['playoff', 'grand final', 'upper', 'lower'])
    if s != current_season:
        if current_season:
            for i in player_elos: player_elos[i] = 1500 + (player_elos[i]-1500)*(1-decay_rate)
            for i in player_p_elos: player_p_elos[i] = 1500 + (player_p_elos[i]-1500)*(1-decay_rate)
        current_season = s
    e_a, e_b = get_elo(b_t, s, player_elos), get_elo(r_t, s, player_elos)
    pe_a, pe_b = get_elo(b_t, s, player_p_elos), get_elo(r_t, s, player_p_elos)
    blue_elos.append(e_a); red_elos.append(e_b); b_p_elos.append(pe_a); r_p_elos.append(pe_b)
    exp_a = 1 / (1 + 10**((e_b-e_a)/400)); act_a = 1 if win == b_t else 0
    k = k_play if is_p else k_reg
    for i in team_rosters.get(b_t, {}).get(s, []): pending_updates[i] = pending_updates.get(i,0) + k*(act_a-exp_a)
    for i in team_rosters.get(r_t, {}).get(s, []): pending_updates[i] = pending_updates.get(i,0) + k*((1-act_a)-(1-exp_a))
    if is_p:
        pexp_a = 1 / (1 + 10**((pe_b-pe_a)/400))
        for i in team_rosters.get(b_t, {}).get(s, []): pending_p_updates[i] = pending_p_updates.get(i,0) + k_play*(act_a-pexp_a)
        for i in team_rosters.get(r_t, {}).get(s, []): pending_p_updates[i] = pending_p_updates.get(i,0) + k_play*((1-act_a)-(1-pexp_a))
    if g_n == last_game.get(m_id):
        for i, d in pending_updates.items(): player_elos[i] = player_elos.get(i, 1500) + d
        for i, d in pending_p_updates.items(): player_p_elos[i] = player_p_elos.get(i, 1500) + d
        pending_updates.clear(); pending_p_updates.clear()

sorted_matches['blue_side_elo_new'] = blue_elos; sorted_matches['red_side_elo_new'] = red_elos
sorted_matches['blue_playoff_elo_new'] = b_p_elos; sorted_matches['red_playoff_elo_new'] = r_p_elos

df_main['match_timestamp'] = pd.to_datetime(df_main['match_timestamp'])
df = df_main.sort_values(['match_timestamp', 'game_number']).reset_index(drop=True)
df = df.merge(sorted_matches[['match_id', 'game_number', 'blue_side_elo_new', 'red_side_elo_new', 'blue_playoff_elo_new', 'red_playoff_elo_new']], on=['match_id', 'game_number'], how='left')
df['blue_side_elo'] = df['blue_side_elo_new'].fillna(df['blue_side_elo'])
df['red_side_elo'] = df['red_side_elo_new'].fillna(df['red_side_elo'])
df['blue_playoff_elo'] = df['blue_playoff_elo_new'].fillna(df['blue_playoff_elo'])
df['red_playoff_elo'] = df['red_playoff_elo_new'].fillna(df['red_playoff_elo'])

Re-computing Elo ratings...


In [3]:
# ----------------------------------------------------
# 3. FEATURE ENGINEERING
# ----------------------------------------------------
print("Engineering features...")
df['momentum_x_side_advantage'] = df['series_momentum_blue'] * df['current_blue_side_advantage']

diff_map = {
    'roster_stability': 'diff_roster_stability', 'patch_practice': 'diff_patch_practice',
    'g3_clutch_wr': 'diff_g3_clutch', 'reverse_sweep_rate': 'diff_reverse_sweep',
    'rest_factor': 'diff_rest', 'playoff_exp': 'diff_playoff_exp',
    'championship_dna': 'diff_championship_dna', 'playoff_winrate': 'diff_playoff_winrate',
    'avg_win_duration': 'diff_avg_win_duration', 'avg_loss_duration': 'diff_avg_loss_duration',
    'lategame_winrate': 'diff_lategame_winrate', 'prev_season_match_wr': 'diff_prev_season_match_wr',
    'prev_season_game_wr': 'diff_prev_season_game_wr', 'prev_season_kda': 'diff_prev_season_kda',
    'draft_exhaustion': 'diff_draft_exhaustion'
}
for base, target in diff_map.items():
    if f'blue_{base}' in df.columns: df[target] = df[f'blue_{base}'] - df[f'red_{base}']
df['diff_rs_rank'] = (df['blue_rs_rank'] - df['red_rs_rank']) * df['is_playoffs']

# Feature lists
base_features_dense = [
    'blue_roster_stability', 'red_roster_stability', 'diff_roster_stability',
    'blue_side_elo', 'red_side_elo', 'blue_playoff_elo', 'red_playoff_elo',
    'blue_championship_dna', 'red_championship_dna', 'blue_playoff_winrate', 'red_playoff_winrate',
    'blue_momentum', 'red_momentum', 'blue_h2h_winrate', 'blue_patch_practice', 'red_patch_practice',
    'diff_patch_practice', 'diff_g3_clutch', 'diff_reverse_sweep', 'diff_rest', 'diff_rs_rank',
    'blue_is_defending_champ', 'red_is_defending_champ', 'is_playoffs', 'blue_g3_clutch_wr', 'red_g3_clutch_wr',
    'blue_reverse_sweep_rate', 'red_reverse_sweep_rate', 'blue_rest_factor', 'red_rest_factor',
    'blue_avg_win_duration', 'red_avg_win_duration', 'current_blue_side_advantage',
    'blue_comfort_patch_score', 'red_comfort_patch_score', 'blue_expected_comfort', 'red_expected_comfort',
    'blue_prev_season_match_wr', 'red_prev_season_match_wr', 'blue_prev_season_game_wr', 'red_prev_season_game_wr',
    'blue_prev_season_kda', 'red_prev_season_kda', 'blue_prev_season_avg_kills', 'red_prev_season_avg_kills',
    'blue_prev_season_avg_deaths', 'red_prev_season_avg_deaths', 'blue_prev_season_avg_assists', 'red_prev_season_avg_assists',
    'diff_prev_season_match_wr', 'diff_prev_season_game_wr', 'diff_prev_season_kda',
]

g1_features = base_features_dense + ['draft_style_sim', 'blue_playoff_exp', 'red_playoff_exp', 'diff_playoff_exp']

series_features = ['series_momentum_blue', 'prev_stomp_margin', 'is_side_swap']
pre_match_series_features = ['blue_draft_exhaustion', 'red_draft_exhaustion', 'diff_draft_exhaustion', 'blue_prev_winner_exhaustion', 'red_prev_winner_exhaustion', 'blue_g1_comfort', 'red_g1_comfort', 'blue_prev_comfort', 'red_prev_comfort']

g2p_features = base_features_dense + series_features + pre_match_series_features + ['draft_style_sim', 'momentum_x_side_advantage', 'blue_playoff_exp', 'red_playoff_exp', 'diff_playoff_exp']

# Split matches chronologically
df = df.sort_values(['match_timestamp', 'game_number']).reset_index(drop=True)
match_start = df.groupby('match_id')['match_timestamp'].min().sort_values(kind='mergesort')
cutoff = match_start.iloc[int(len(match_start)*0.85)]
train_df = df[df['match_id'].isin(match_start[match_start < cutoff].index)].copy()
test_df = df[df['match_id'].isin(match_start[match_start >= cutoff].index)].copy()

Engineering features...


In [4]:
# ----------------------------------------------------
# 4. MODEL TRAINING, EVALUATION, AND ADVANCED VALIDATION
# ----------------------------------------------------
import pickle
from sklearn.metrics import log_loss, brier_score_loss

print("Training Game 1 XGBoost...")
g1_train = train_df[train_df['game_number'] == 1]; g1_test = test_df[test_df['game_number'] == 1]
g1_model = xgb.XGBClassifier(n_estimators=400, max_depth=2, learning_rate=0.02, subsample=0.9, colsample_bytree=0.8, reg_lambda=2.0, random_state=42, eval_metric='logloss', verbosity=0)
g1_model.fit(g1_train[g1_features], g1_train['target_blue_win'])

print("Training Game 2+ Ensemble (CatBoost + RF Blend)...")
g2p_train = train_df[train_df['game_number'] > 1]; g2p_test = test_df[test_df['game_number'] > 1]
c1 = CatBoostClassifier(iterations=100, depth=4, learning_rate=0.05, subsample=0.9, l2_leaf_reg=3.0, random_seed=42, verbose=0)
rf = RandomForestClassifier(n_estimators=200, max_depth=2, random_state=42)
c2 = CatBoostClassifier(iterations=50, depth=4, learning_rate=0.05, subsample=0.9, l2_leaf_reg=3.0, random_seed=42, verbose=0)

for m in [c1, rf, c2]: m.fit(g2p_train[g2p_features], g2p_train['target_blue_win'], sample_weight=g2p_train['time_weight'])

# Serialize and save trained models
import os
os.makedirs("models", exist_ok=True)
with open("models/g1_xgb.pkl", "wb") as f:
    pickle.dump(g1_model, f)
with open("models/g2p_cat1.pkl", "wb") as f:
    pickle.dump(c1, f)
with open("models/g2p_rf.pkl", "wb") as f:
    pickle.dump(rf, f)
with open("models/g2p_cat2.pkl", "wb") as f:
    pickle.dump(c2, f)
print("Models serialized and saved successfully to 'models/'.")

# Final Evaluation
p1 = g1_model.predict_proba(g1_test[g1_features])[:, 1]
p2 = (1*c1.predict_proba(g2p_test[g2p_features])[:,1] + 3*rf.predict_proba(g2p_test[g2p_features])[:,1] + 2*c2.predict_proba(g2p_test[g2p_features])[:,1]) / 6

acc_g1 = accuracy_score(g1_test['target_blue_win'], (p1>=0.5).astype(int))
acc_g2 = accuracy_score(g2p_test['target_blue_win'], (p2>=0.5).astype(int))

y_true_comb = pd.concat([g1_test['target_blue_win'], g2p_test['target_blue_win']])
y_prob_comb = np.concatenate([p1, p2])
y_pred_comb = (y_prob_comb >= 0.5).astype(int)
comb_acc = accuracy_score(y_true_comb, y_pred_comb)

# Calculate advanced metrics
loss_g1 = log_loss(g1_test['target_blue_win'], p1)
loss_g2 = log_loss(g2p_test['target_blue_win'], p2)
loss_comb = log_loss(y_true_comb, y_prob_comb)

brier_g1 = brier_score_loss(g1_test['target_blue_win'], p1)
brier_g2 = brier_score_loss(g2p_test['target_blue_win'], p2)
brier_comb = brier_score_loss(y_true_comb, y_prob_comb)

print("\n" + "="*50)
print(f"Game 1 Accuracy: {acc_g1:.2%}  | Log Loss: {loss_g1:.4f} | Brier: {brier_g1:.4f}")
print(f"Game 2+ Accuracy: {acc_g2:.2%} | Log Loss: {loss_g2:.4f} | Brier: {brier_g2:.4f}")
print(f"FINAL COMBINED ACCURACY: {comb_acc:.2%} | Log Loss: {loss_comb:.4f} | Brier: {brier_comb:.4f}")
print("="*50)

# Print calibration profile
def print_calibration_stats(y_true, y_prob, name):
    print(f"\nCalibration Profile for {name}:")
    bins = [(0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 1.0)]
    for low, high in bins:
        pred_probs = []
        pred_correct = []
        for yt, yp in zip(y_true, y_prob):
            conf = yp if yp >= 0.5 else 1 - yp
            pred_win = 1 if yp >= 0.5 else 0
            if low <= conf < high:
                pred_probs.append(conf)
                pred_correct.append(1 if pred_win == yt else 0)
        if pred_correct:
            print(f"  Confidence [{low:.1f} - {high:.1f}): Games={len(pred_correct)}, Pred Prob={np.mean(pred_probs):.2%}, Actual Acc={np.mean(pred_correct):.2%}")
        else:
            print(f"  Confidence [{low:.1f} - {high:.1f}): No games")

print_calibration_stats(g1_test['target_blue_win'], p1, "Game 1")
print_calibration_stats(g2p_test['target_blue_win'], p2, "Game 2+")
print_calibration_stats(y_true_comb, y_prob_comb, "Combined")

# Rolling Season-by-Season Backtest
print("\n" + "="*50)
print("ROLLING SEASON-BY-SEASON BACKTEST")
print("="*50)
backtest_seasons = [13, 14, 15, 16, 17]
backtest_results = []

for target_season in backtest_seasons:
    bt_train = df[df['season'] < target_season].copy()
    bt_test = df[df['season'] == target_season].copy()
    if bt_train.empty or bt_test.empty: continue
    
    bt_g1_train = bt_train[bt_train['game_number'] == 1]
    bt_g1_test = bt_test[bt_test['game_number'] == 1]
    
    bt_g1_model = xgb.XGBClassifier(n_estimators=400, max_depth=2, learning_rate=0.02, subsample=0.9, colsample_bytree=0.8, reg_lambda=2.0, random_state=42, eval_metric='logloss', verbosity=0)
    bt_g1_model.fit(bt_g1_train[g1_features], bt_g1_train['target_blue_win'])
    
    bt_g2p_train = bt_train[bt_train['game_number'] > 1]
    bt_g2p_test = bt_test[bt_test['game_number'] > 1]
    
    bt_c1 = CatBoostClassifier(iterations=100, depth=4, learning_rate=0.05, subsample=0.9, l2_leaf_reg=3.0, random_seed=42, verbose=0)
    bt_rf = RandomForestClassifier(n_estimators=200, max_depth=2, random_state=42)
    bt_c2 = CatBoostClassifier(iterations=50, depth=4, learning_rate=0.05, subsample=0.9, l2_leaf_reg=3.0, random_seed=42, verbose=0)
    
    for m in [bt_c1, bt_rf, bt_c2]:
        m.fit(bt_g2p_train[g2p_features], bt_g2p_train['target_blue_win'], sample_weight=bt_g2p_train['time_weight'])
        
    p1_bt = bt_g1_model.predict_proba(bt_g1_test[g1_features])[:, 1] if not bt_g1_test.empty else np.array([])
    p2_bt = (1*bt_c1.predict_proba(bt_g2p_test[g2p_features])[:,1] + 3*bt_rf.predict_proba(bt_g2p_test[g2p_features])[:,1] + 2*bt_c2.predict_proba(bt_g2p_test[g2p_features])[:,1]) / 6 if not bt_g2p_test.empty else np.array([])
    
    acc_g1_bt = accuracy_score(bt_g1_test['target_blue_win'], (p1_bt>=0.5).astype(int)) if len(p1_bt) > 0 else 0.0
    acc_g2_bt = accuracy_score(bt_g2p_test['target_blue_win'], (p2_bt>=0.5).astype(int)) if len(p2_bt) > 0 else 0.0
    
    y_true_comb_bt = pd.concat([bt_g1_test['target_blue_win'], bt_g2p_test['target_blue_win']])
    y_pred_comb_bt = np.concatenate([(p1_bt>=0.5).astype(int), (p2_bt>=0.5).astype(int)])
    comb_acc_bt = accuracy_score(y_true_comb_bt, y_pred_comb_bt) if len(y_true_comb_bt) > 0 else 0.0
    
    print(f"Season {target_season:02d}: G1 Acc = {acc_g1_bt:.2%}, G2+ Acc = {acc_g2_bt:.2%} | Combined Acc = {comb_acc_bt:.2%}")
    backtest_results.append(comb_acc_bt)

if backtest_results:
    print(f"\nAverage Sequential Backtest Accuracy: {np.mean(backtest_results):.2%}")

Training Game 1 XGBoost...


Training Game 2+ Ensemble (CatBoost + RF Blend)...


Models serialized and saved successfully to 'models/'.

Game 1 Accuracy: 75.91%  | Log Loss: 0.5854 | Brier: 0.1979
Game 2+ Accuracy: 57.87% | Log Loss: 0.6760 | Brier: 0.2410
FINAL COMBINED ACCURACY: 64.52% | Log Loss: 0.6427 | Brier: 0.2251

Calibration Profile for Game 1:
  Confidence [0.5 - 0.6): Games=49, Pred Prob=54.95%, Actual Acc=69.39%
  Confidence [0.6 - 0.7): Games=51, Pred Prob=65.50%, Actual Acc=84.31%
  Confidence [0.7 - 0.8): Games=32, Pred Prob=73.84%, Actual Acc=71.88%
  Confidence [0.8 - 1.0): Games=5, Pred Prob=81.86%, Actual Acc=80.00%

Calibration Profile for Game 2+:
  Confidence [0.5 - 0.6): Games=114, Pred Prob=54.91%, Actual Acc=51.75%
  Confidence [0.6 - 0.7): Games=86, Pred Prob=65.19%, Actual Acc=65.12%
  Confidence [0.7 - 0.8): Games=35, Pred Prob=72.95%, Actual Acc=60.00%
  Confidence [0.8 - 1.0): No games

Calibration Profile for Combined:
  Confidence [0.5 - 0.6): Games=163, Pred Prob=54.92%, Actual Acc=57.06%
  Confidence [0.6 - 0.7): Games=137, Pred P

Season 13: G1 Acc = 54.69%, G2+ Acc = 69.79% | Combined Acc = 63.75%


Season 14: G1 Acc = 65.62%, G2+ Acc = 57.94% | Combined Acc = 60.82%


Season 15: G1 Acc = 62.50%, G2+ Acc = 63.64% | Combined Acc = 63.22%


Season 16: G1 Acc = 75.00%, G2+ Acc = 57.14% | Combined Acc = 63.64%


Season 17: G1 Acc = 76.79%, G2+ Acc = 61.54% | Combined Acc = 67.91%

Average Sequential Backtest Accuracy: 63.87%


In [5]:
# ----------------------------------------------------
# 5. POST-DRAFT BENCHMARK (FOR ANALYSIS ONLY)
# ----------------------------------------------------
print("Evaluating Post-Draft Benchmark...")
draft_features = ['blue_synergy', 'red_synergy', 'blue_counter', 'red_counter', 'blue_buffs_in_draft', 'blue_nerfs_in_draft', 'red_buffs_in_draft', 'red_nerfs_in_draft']
X_pd_train = g2p_train[g2p_features + draft_features]
X_pd_test = g2p_test[g2p_features + draft_features]
pd_model = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
pd_model.fit(X_pd_train, g2p_train['target_blue_win'], sample_weight=g2p_train['time_weight'])
pd_acc = accuracy_score(g2p_test['target_blue_win'], pd_model.predict(X_pd_test))
print(f"Post-Draft Game 2+ Accuracy: {pd_acc:.2%} (vs {acc_g2:.2%} pre-match)")

Evaluating Post-Draft Benchmark...
Post-Draft Game 2+ Accuracy: 60.00% (vs 57.87% pre-match)


In [6]:
# ----------------------------------------------------
# 6. SERIES SIMULATOR
# ----------------------------------------------------
def get_recent_stats(team_name):
    t = resolve_team_name(team_name)
    team_games = df[(df['blue_side_team']==t)|(df['red_side_team']==t)]
    if team_games.empty: return None
    last = team_games.iloc[-1]
    side = 'blue' if last['blue_side_team'] == t else 'red'
    return {col.replace(f'{side}_', ''): last[col] for col in df.columns if col.startswith(f'{side}_')}

def predict_matchup(blue_team, red_team, game_num=1, b_won_last=None):
    b_stats, r_stats = get_recent_stats(blue_team), get_recent_stats(red_team)
    if not b_stats or not r_stats: return "Team not found"
    row = {}
    for f in g2p_features:
        if f.startswith('diff_'): base = f.replace('diff_', ''); row[f] = b_stats.get(base, 0) - r_stats.get(base, 0)
        elif f.startswith('blue_'): row[f] = b_stats.get(f.replace('blue_', ''), 0)
        elif f.startswith('red_'): row[f] = r_stats.get(f.replace('red_', ''), 0)
        else: row[f] = 0
    row['series_momentum_blue'] = 0.5 if b_won_last is None else (1.0 if b_won_last else 0.0)
    row['momentum_x_side_advantage'] = row['series_momentum_blue'] * df['current_blue_side_advantage'].iloc[-1]
    row['is_playoffs'] = 1
    inp = pd.DataFrame([row])
    if game_num == 1: pr = g1_model.predict_proba(inp[g1_features])[0, 1]
    else: pr = (1*c1.predict_proba(inp[g2p_features])[0,1] + 3*rf.predict_proba(inp[g2p_features])[0,1] + 2*c2.predict_proba(inp[g2p_features])[0,1]) / 6
    print(f"{blue_team} ({pr:.1%}) vs {red_team} ({1-pr:.1%}) -> Predicted: {'Blue' if pr>0.5 else 'Red'}")

print("\nSample Predictions:")
predict_matchup("Team Liquid PH", "Team Falcons PH")
predict_matchup("ONIC PH", "RSG PH")


Sample Predictions:
Team Liquid PH (44.2%) vs Team Falcons PH (55.8%) -> Predicted: Red
ONIC PH (62.0%) vs RSG PH (38.0%) -> Predicted: Blue
